In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.cluster import KMeans

import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully.')

In [ ]:
np.random.seed(42)
n = 100

def generate_species(sepal_l, sepal_w, petal_l, petal_w, std=0.15):
    return np.column_stack([
        np.random.normal(sepal_l, std, n),
        np.random.normal(sepal_w, std, n),
        np.random.normal(petal_l, std * 1.5, n),
        np.random.normal(petal_w, std, n),
    ])

species_params = {
    'Iris Setosa':     (5.0, 3.4, 1.5, 0.3),
    'Iris Versicolor': (5.9, 2.8, 4.3, 1.3),
    'Iris Virginica':  (6.6, 3.0, 5.6, 2.0),
    'Rose':            (4.2, 2.5, 6.5, 3.8),
    'Tulip':           (5.5, 3.8, 7.2, 2.2),
    'Sunflower':       (8.5, 4.5, 12.0, 5.5),
}

frames = []
for name, params in species_params.items():
    data = generate_species(*params)
    df_s = pd.DataFrame(data, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
    df_s['species'] = name
    frames.append(df_s)

df = pd.concat(frames, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print('Dataset shape:', df.shape)
print('\nClass distribution:')
print(df['species'].value_counts())
df.head()

In [ ]:
print('Statistical Summary:')
df.describe()

In [ ]:
print('Missing values:', df.isnull().sum().sum())

In [ ]:
COLORS = {
    'Iris Setosa': '#6aab63', 'Iris Versicolor': '#a87d50', 'Iris Virginica': '#7b6fa0',
    'Rose': '#b85555', 'Tulip': '#c47a3a', 'Sunflower': '#c8a820'
}

features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for i, (ax, feat) in enumerate(zip(axes.flat, features)):
    for species, color in COLORS.items():
        ax.hist(df[df['species'] == species][feat], bins=15, alpha=0.5, color=color, label=species)
    ax.set_title(feat.replace('_', ' ').title())
    ax.set_xlabel('cm')
    ax.set_ylabel('Frequency')
    if i == 0:
        ax.legend(fontsize=7)

plt.suptitle('Feature Distributions by Species', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlOrBr', square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 5))
for ax, feat in zip(axes, features):
    data_by_species = [df[df['species'] == s][feat].values for s in COLORS]
    bp = ax.boxplot(data_by_species, patch_artist=True)
    for patch, color in zip(bp['boxes'], COLORS.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels([s.split()[-1] for s in COLORS], rotation=30, ha='right', fontsize=7)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=9)

plt.suptitle('Boxplots by Species', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
X = df[features].values
y = df['species'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train_scaled, y_train)
print('Model trained.')

In [ ]:
y_pred = model.predict(X_test_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%\n')
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred, labels=list(COLORS.keys()))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=list(COLORS.keys()), yticklabels=list(COLORS.keys()))
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(7, 4))
plt.bar(range(4), importances[indices], color='#4a7c45')
plt.xticks(range(4), [features[i].replace('_', ' ').title() for i in indices], rotation=15)
plt.title('Feature Importances', fontsize=13, fontweight='bold')
plt.ylabel('Importance')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
inertias = []
for k in range(1, 12):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, 12), inertias, 'o-', color='#4a7c45', linewidth=2)
plt.axvline(x=6, color='#c8b560', linestyle='--', label='Optimal k=6')
plt.title('Elbow Method', fontsize=13, fontweight='bold')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.legend()
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X[:, 2], X[:, 3], c=cluster_labels, cmap='tab10', alpha=0.6)
axes[0].set_xlabel('Petal Length (cm)')
axes[0].set_ylabel('Petal Width (cm)')
axes[0].set_title('K-Means Clusters (k=6)')

label_nums = pd.Categorical(y).codes
axes[1].scatter(X[:, 2], X[:, 3], c=label_nums, cmap='tab10', alpha=0.6)
axes[1].set_xlabel('Petal Length (cm)')
axes[1].set_ylabel('Petal Width (cm)')
axes[1].set_title('Actual Species Labels')

plt.suptitle('Clustering vs Actual Labels', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('clustering.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
joblib.dump(model, 'flower_model.joblib')
joblib.dump(scaler, 'scaler.joblib')
print('Saved: flower_model.joblib, scaler.joblib')

In [ ]:
loaded_model  = joblib.load('flower_model.joblib')
loaded_scaler = joblib.load('scaler.joblib')

sample = np.array([[8.5, 4.5, 12.0, 5.5]])
pred   = loaded_model.predict(loaded_scaler.transform(sample))[0]
conf   = loaded_model.predict_proba(loaded_scaler.transform(sample)).max() * 100
print(f'Test prediction: {pred} ({conf:.1f}% confidence)')